In [1]:
import pandas as pd
import re
import numpy as np
from scipy.stats import mannwhitneyu
import krippendorff
from collections import Counter
from scipy.stats import wilcoxon
from utils.database import Database
import json

/path/to/folder/.conda/envs/attributeexpl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'/path/to/folder/Qwen2.5-7B-Instruct': [[0, 1, 2], [1, 2, 3], [2, 3, 4], [3, 4, 5], [4, 5, 6], [5, 6, 7], [6, 7, 8], [7, 8, 9], [8, 9, 10], [9, 10, 11], [10, 11, 12], [11, 12, 13], [12, 13, 14], [13, 14, 15], [14, 15, 16], [15, 16, 17], [16, 17, 18], [17, 18, 19], [18, 19, 20], [19, 20, 21], [20, 21, 22], [21, 22, 23], [22, 23, 24], [23, 24, 25], [24, 25, 26], [25, 26, 27]], '/path/to/folder/Llama-3.1-8B-Instruct': [[0, 1, 2], [1, 2, 3], [2, 3, 4], [3, 4, 5], [4, 5, 6], [5, 6, 7], [6, 7, 8], [7, 8, 9], [8, 9, 10], [9, 10, 11], [10, 11, 12], [11, 12, 13], [12, 13, 14], [13, 14, 15], [14, 15, 16], [15, 16, 17], [16, 17, 18], [17, 18, 19], [18, 19, 20], [19, 20, 21], [20, 21, 22], [21, 22, 23], [22, 23, 24], [23, 24, 25], [24, 25, 26], [25, 26, 27], [26, 27, 28], [27, 28, 29], [28, 29, 30], [29, 30, 31]], '/path/to/folder/Ministral-8B-Instruct-2410': [[0, 1, 2], [1, 2, 3], [2, 3, 4], [3, 4, 5], [4, 5, 6], [5, 6, 7], [6, 7, 8], [7, 8, 9], [8, 9, 10], [9, 10, 11], [10, 11, 12], [11, 12, 13

In [2]:
def harmonic_mean_two(criterion1, criterion2):
    hmean = (2 * criterion1 * criterion2) / (criterion1 + criterion2)
    return hmean

In [3]:
STEERING_METRICS = (
    "steering_effect_lisa",
    "direction_error_lisa",
    "possible_steering_effect_lisa",
    "possible_improvement_lisa",
    "possible_improvement_with_penalty_lisa",
    "steering_effect_normal",
    "direction_error_normal",
    "possible_steering_effect_normal",
    "semantic_alignment_normal",
    "semantic_alignment_with_penalty_normal",
    "group_score",
    "group_score_with_penalty"
    #"factual_accuracy",
)

In [4]:
PROMPT_COMPONENTS = {
    "prompt_group": "Group",
    "prompt_group_attribute_3": "Group \\textbf{+ attributes (3)}",
    "prompt_group_attribute_5": "Group \\textbf{+ attributes (5)}",
    "prompt_group_attribute_10": "Group \\textbf{+ attributes (10)}",

    "activation_group_3": "Group",
    "activation_group_attribute_3": "Group \\textbf{+ attributes (3)}",
    "activation_group_5": "Group",
    "activation_group_attribute_5": "Group \\textbf{+ attributes (5)}",
    "activation_group_10": "Group",
    "activation_group_attribute_10": "Group \\textbf{+ attributes (10)}",

    "activation_konen_group": "Group",
    "activation_konen_group_attribute_3": "Group \\textbf{+ attributes (3)}",
    "activation_konen_group_attribute_5": "Group \\textbf{+ attributes (5)}",
    "activation_konen_group_attribute_10": "Group \\textbf{+ attributes (10)}",
}

In [5]:
STEERING_VECTORS = {
    "prompt_group": "-",
    "prompt_group_attribute_5": "-",
    "prompt_group_attribute_10": "-",

    "activation_group_3": "\\textbf{Our approach (3)}",
    "activation_group_attribute_3": "\\textbf{Our approach (3)}",
    "activation_group_5": "\\textbf{Our approach (5)}",
    "activation_group_attribute_5": "\\textbf{Our approach (5)}",
    "activation_group_10": "\\textbf{Our approach (10)}",
    "activation_group_attribute_10": "\\textbf{Our approach (10)}",

    "activation_konen_group": "\\citet{konen2024steeringstyle}",
    "activation_konen_group_attribute_5": "\\citet{konen2024steeringstyle}",
}

In [6]:
#steering_model="/path/to/folder/Llama-3.1-8B-Instruct"
#DATABASE_NAME="llama_data_qwen_filtered_sent.sqlite"
#steering_model = "/path/to/folder/Qwen2.5-7B-Instruct"
#DATABASE_NAME = "qwen_data_qwen_filtered_sent.sqlite"
#DATABASE_NAME = "qwen_3_data_qwen_filtered_sent.sqlite"
#DATABASE_NAME = "qwen_10_data_qwen_filtered_sent.sqlite"
#steering_model = "/path/to/folder/Qwen2.5-32B-Instruct"
#DATABASE_NAME = "qwen_big_data_qwen_filtered_sent.sqlite"
steering_model = "/path/to/folder/Ministral-8B-Instruct-2410"
DATABASE_NAME = f"mistral_data_qwen_filtered_sent.sqlite"
database = Database(DATABASE_NAME)
database.create_database()

In [ ]:
steered_data = database.get_steering_generation_of_llm(steering_model)
steering_types_order = steered_data["steering_type"].unique().sort().remove("base")
print("Steering types:", steering_types_order)

best_activation_hyperparameters = {}
for steering_type in steering_types_order:
    row = steered_data.loc[
        (steered_data["steering_type"] == steering_type) &
        (steered_data["question_id"] == 717)
    ].squeeze()


    best_activation_layer = list(map(int, row["activation_layer"].split(",")))
    best_activation_lambda = row["activation_lambda"]

    best_activation_hyperparameters[steering_type] = {"layer_indices": best_activation_layer, "lambda": best_activation_lambda}

print("Best hyperparameters:", best_activation_hyperparameters)

In [7]:
steering_types_order = ["activation_group_5", "activation_group_attribute_5", "activation_konen_group", "activation_konen_group_attribute_5", "prompt_group", "prompt_group_attribute_5"]
#steering_types_order = ["activation_group_3", "activation_group_attribute_3", "activation_konen_group_attribute_3", "prompt_group_attribute_3"]
#steering_types_order = ["activation_group_10", "activation_group_attribute_10", "activation_konen_group_attribute_10", "prompt_group_attribute_10"]
#steering_types_order = ["activation_group_10", "activation_group_attribute_10", "prompt_group", "prompt_group_attribute_10"]

best_activation_hyperparameters = {}
#best_activation_hyperparameters["activation_group_10"] = {"layer_indices": [11,12,13], "lambda": 2.0}

In [8]:
test_data_split = "rooein_test"
steering_result = np.load(f"results_qwen_old_prompt/np_steering_results_{test_data_split}_{database.file_name}.npy")
group_score_result = steering_result[STEERING_METRICS.index("group_score")].reshape(len(steering_types_order), -1)

with open(f"results_qwen_old_prompt/results_factscore_{database.file_name}.json", "r") as f:
    factscore_result = json.load(f)

In [9]:
results = {}
for steering_type, group_specificity in zip(steering_types_order, group_score_result):
    if steering_type not in factscore_result:
        continue
    results[steering_type] = {"prompt_component": PROMPT_COMPONENTS[steering_type],
                              "steering_vectors": STEERING_VECTORS[steering_type],
                              "lambda": 0,#best_activation_hyperparameters[steering_type]["lambda"],
                              "layers": 0,#best_activation_hyperparameters[steering_type]["layer_indices"],
                              "specificity": np.mean(group_specificity),
                              "factuality": np.mean(factscore_result[steering_type]["factscores_cleaned"]),
                              "hmean": harmonic_mean_two(np.mean(group_specificity), np.mean(factscore_result[steering_type]["factscores_cleaned"]))}

In [10]:
for steering_type in factscore_result:#["activation_group_3", "activation_group_attribute_3", "activation_group_10", "activation_group_attribute_10"]:
    if steering_type in results:        
        row = (f"& {results[steering_type]['prompt_component']}",
                  f" & {results[steering_type]['steering_vectors']}"
                  f" & {results[steering_type]['lambda']}"
                  f" & {results[steering_type]['layers']}"
                  f" & {results[steering_type]['specificity']:.3f}"
                  f" & {results[steering_type]['factuality']:.3f}"
                  f" & {results[steering_type]['hmean']:.3f} \\\\")
        print(" ".join(row))

& Group  & - & 0 & 0 & 0.111 & 0.376 & 0.171 \\
& Group \textbf{+ attributes (5)}  & - & 0 & 0 & 0.191 & 0.346 & 0.246 \\
& Group  & \citet{konen2024steeringstyle} & 0 & 0 & 0.300 & 0.252 & 0.274 \\
& Group \textbf{+ attributes (5)}  & \citet{konen2024steeringstyle} & 0 & 0 & 0.511 & 0.187 & 0.274 \\
& Group  & \textbf{Our approach (5)} & 0 & 0 & 0.128 & 0.354 & 0.188 \\
& Group \textbf{+ attributes (5)}  & \textbf{Our approach (5)} & 0 & 0 & 0.215 & 0.330 & 0.260 \\


In [13]:
# Siginificance test
results = {}
for steering_type, group_specificity in zip(steering_types_order, group_score_result):
    if steering_type not in factscore_result:
        continue
    results[steering_type] = {"specificity": group_specificity,
                              "factuality": np.array(factscore_result[steering_type]["factscores_cleaned"]),
                             }

for metric in ["specificity", "factuality"]:
    print(f"Signifinance test for {metric}:\n")
    for prompt_component in ["group", "group_attribute_5"]:
        if prompt_component == "group":
            results_approach = results[f"activation_{prompt_component}_5"][metric]
        else:
            results_approach = results[f"activation_{prompt_component}"][metric]
        for baseline in ["prompt", "activation_konen"]:
            results_baseline = results[f"{baseline}_{prompt_component}"][metric]
            if np.mean(results_approach) > np.mean(results_baseline):
                alternative = "greater"
            else:
                continue
                #alternative = "less"
            print(f"Setting: activation_{prompt_component} ({np.mean(results_approach)}) vs. {baseline}_{prompt_component} ({np.mean(results_baseline)})")   
            diff = results_approach - results_baseline

            # Check that differences are normally distributed
            from scipy.stats import shapiro
            stat, p = shapiro(diff)
            print("Normal distribution test p-value:", p)
            assert p < 0.05

            # Check that differences are symmetric with skewness
            from scipy.stats import skew
            skewness = skew(diff)
            print("Skewness:", skewness)

            # Wilcoxon signed-rank test
            from scipy.stats import wilcoxon
            #stat, p_wilcoxon = wilcoxon(results_approach,
            #                            results_baseline,
            #                            alternative='two-sided')
            #print("Wilcoxon signed-rank test (two-sided) p-value:", p_wilcoxon)
            stat, p_wilcoxon = wilcoxon(results_approach,
                                        results_baseline,
                                        alternative=alternative)
            print(f"Wilcoxon signed-rank test ({alternative}) p-value:", p_wilcoxon)

            # Permutation test
            #from scipy.stats import permutation_test
            #def mean_diff(a, b):
            #    return np.mean(a - b)
            #res = permutation_test(
            #    (results_approach, results_baseline),
            #    statistic=mean_diff,
            #    vectorized=False,
            #    n_resamples=100000,  # large number for stability
            #    alternative='two-sided'
            #)
            #print("Permutation test (two-sided') p-value:", res.pvalue)
            #res = permutation_test(
            #    (results_approach, results_baseline),
            #    statistic=mean_diff,
            #    vectorized=False,
            #    n_resamples=100000,  # large number for stability
            #    alternative=alternative
            #)
            #print(f"Permutation test ({alternative}) p-value:", res.pvalue)
            print()

Signifinance test for specificity:

Setting: activation_group (0.1281200647354126) vs. prompt_group (0.1105564683675766)
Normal distribution test p-value: 4.258388171721931e-49
Skewness: 2.583246946334839
Wilcoxon signed-rank test (greater) p-value: 1.7711198547147407e-06

Setting: activation_group_attribute_5 (0.2146095335483551) vs. prompt_group_attribute_5 (0.19097310304641724)
Normal distribution test p-value: 4.097491578009158e-41
Skewness: 0.6541529893875122
Wilcoxon signed-rank test (greater) p-value: 1.4583199913146065e-06

Signifinance test for factuality:

Setting: activation_group (0.35360236698170017) vs. activation_konen_group (0.25161921558218425)
Normal distribution test p-value: 3.03285537471728e-08
Skewness: 0.29191615414612604
Wilcoxon signed-rank test (greater) p-value: 1.7889009209743698e-42

Setting: activation_group_attribute_5 (0.33027539878148204) vs. activation_konen_group_attribute_5 (0.18740605262497717)
Normal distribution test p-value: 8.589042282879083e-07

In [17]:
#steering_types_order = ["activation_group_5", "activation_group_attribute_5", "activation_konen_group", "activation_konen_group_attribute_5", "prompt_group", "prompt_group_attribute_5"]
#steering_types_order = ["activation_group_3", "activation_group_attribute_3", "activation_konen_group_attribute_3", "prompt_group_attribute_3"]
#steering_types_order = ["activation_group_10", "activation_group_attribute_10", "activation_konen_group_attribute_10", "prompt_group_attribute_10"]
steering_types_order = ["activation_group_10", "activation_group_attribute_10", "prompt_group", "prompt_group_attribute_10"]

best_activation_hyperparameters = {}
#best_activation_hyperparameters["activation_group_10"] = {"layer_indices": [11,12,13], "lambda": 2.0}

In [18]:
test_data_split = "eli5_test"
steering_result = np.load(f"results_qwen_old_prompt/np_steering_results_{test_data_split}_{database.file_name}.npy")
group_score_result = steering_result[STEERING_METRICS.index("group_score")].reshape(len(steering_types_order), -1)

with open(f"results_qwen_old_prompt/results_factscore_{database.file_name}.json", "r") as f:
    factscore_result = json.load(f)

In [19]:
results = {}
for steering_type, group_specificity in zip(steering_types_order, group_score_result):
    if steering_type not in factscore_result:
        continue
    results[steering_type] = {"prompt_component": PROMPT_COMPONENTS[steering_type],
                              "steering_vectors": STEERING_VECTORS[steering_type],
                              "lambda": 0,#best_activation_hyperparameters[steering_type]["lambda"],
                              "layers": 0,#best_activation_hyperparameters[steering_type]["layer_indices"],
                              "specificity": np.mean(group_specificity)}

In [20]:
for steering_type in factscore_result:#["activation_group_3", "activation_group_attribute_3", "activation_group_10", "activation_group_attribute_10"]:
    if steering_type in results:        
        row = (f"& {results[steering_type]['prompt_component']}",
                  f" & {results[steering_type]['steering_vectors']}"
                  f" & {results[steering_type]['lambda']}"
                  f" & {results[steering_type]['layers']}"
                  f" & {results[steering_type]['specificity']:.3f} \\\\")
        print(" ".join(row))

& Group  & - & 0 & 0 & 0.157 \\
& Group \textbf{+ attributes (10)}  & - & 0 & 0 & 0.256 \\
& Group  & \textbf{Our approach (10)} & 0 & 0 & 0.213 \\
& Group \textbf{+ attributes (10)}  & \textbf{Our approach (10)} & 0 & 0 & 0.480 \\
